# 5.3 Lab: Cross-Validation and the Bootstrap

In this lab we explore the resampling techniques covered in Chapter 5. We begin by importing all needed dependencies at the top level.

In [2]:
import numpy as np
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import ModelSpec as MS, summarize, poly
from sklearn.model_selection import train_test_split

from functools import partial
from sklearn.model_selection import cross_validate, KFold, ShuffleSplit
from sklearn.base import clone
from ISLP.models import sklearn_sm

## 5.3.1 The Validation Set Approach

For this portion of the lab we focus on the `Auto` data set. We use the function `train_test_split()` to split the data into training and validation sets. There are 392 observations, so we split into two equal sizes of 196 observations using the argument `test_size=196`. We also set the `random_seed` keyword argument for reproducability of results.

In [3]:
Auto = load_data('Auto')
Auto_train, Auto_valid = train_test_split(Auto, test_size=196, random_state=0)

We fit a linear regression using only the observations from the training set, `Auto_train`.

In [4]:
hp_mm = MS(['horsepower'])
X_train = hp_mm.fit_transform(Auto_train)
y_train = Auto_train['mpg']
model = sm.OLS(y_train, X_train)
results = model.fit()

We use `predict()` method of `results` for the model using the validation data set. We also calculate the mean-squared error (MSE) for the predicted values.

In [10]:
X_valid = hp_mm.transform(Auto_valid)
y_valid = Auto_valid['mpg']
valid_pred = results.predict(X_valid)
print(f"{np.mean((y_valid - valid_pred)**2):.4f}")

23.6166


Hence our estimate for the validation MSE for the linear regression (i.e., the true MSE) is 23.62.

We can do this again for higher-degree polynomial regression. We write the following function to do so.

In [11]:
def evalMSE(terms, response, train, test):

    mm = MS(terms)
    X_train = mm.fit_transform(train)
    y_train = train[response]

    X_test = mm.transform(test)
    y_test = test[response]

    results = sm.OLS(y_train, X_train).fit()
    test_pred = results.predict(X_test)

    return np.mean((y_test - test_pred)**2)

We run a test for the linear, quadratic, and cubic fits.

In [13]:
MSE = np.zeros(3)
for idx, degree in enumerate(range(1,4)):
    MSE[idx] = evalMSE([poly('horsepower', degree)], 'mpg', Auto_train, Auto_valid)

MSE

array([23.61661707, 18.76303135, 18.79694163])

We see a decrease in MSE from the linear to the quadratic, aligning with previous repsonses that `mpg` can be more accurately modeled using $\texttt{horsepower}$ and $\texttt{horsepower}^2$ than just $\texttt{horsepower}$. However, including a cubic term does not introduce any meaningful gains.

Next we observe what happens when we change the `random_state`, resulting in a change to our training and validation sets.

In [14]:
Auto_train, Auto_valid = train_test_split(Auto, test_size=196, random_state=3)

MSE = np.zeros(3)
for idx, degree in enumerate(range(1,4)):
    MSE[idx] = evalMSE([poly('horsepower', degree)], 'mpg', Auto_train, Auto_valid)

MSE

array([20.75540796, 16.94510676, 16.97437833])

Even though our training and validation sets are different, we still see the reduction in validation MSE from linear to quadratic, but not from quadratic to cubic.

## 5.3.2 Cross-Validation

While cross-validation (CV) can be performed for any GLM, the `sklearn` package has the simplist way to implement CV. While the primary package used for modeling in this text is `statsmodel`, the `ISLP` package provides a wrapper so that the `statsmodel` approaches can leverage the CV algorithms within `sklearn`. The wrapper is `sklearn_sm`. The following is an example of using the wrapper with LOOCV.

In [20]:
hp_model = sklearn_sm(model_type=sm.OLS, model_spec=MS(['horsepower']))
X, Y = Auto.drop(columns=['mpg']), Auto['mpg']
cv_results = cross_validate(hp_model, X, Y, cv=Auto.shape[0])
cv_err = np.mean(cv_results['test_score'])
print(f"{cv_err:.3f}")

24.232


Providing the `cv` keyword argument to `cross_validate` informs the function how many folds to use. Submitting the full size of the dataset results in $n$ folds, i.e., LOOCV. While the `cv_results` variable contains a dictionary of several components, we only want the test MSE in the above example.

Next we repeat the above process but with increasingly complex polynomial fits.

In [25]:
cv_error = np.zeros(5)
H = np.array(Auto['horsepower'])
M = sklearn_sm(sm.OLS)
for i, d in enumerate(range(1,6)):
    X = np.power.outer(H, np.arange(d+1))
    M_CV = cross_validate(M, X, Y, cv=Auto.shape[0])
    cv_error[i] = f"{np.mean(M_CV['test_score']):.4f}"
cv_error

array([24.2315, 19.2482, 19.335 , 19.4244, 19.0332])

Once again we see a clear reduction in test MSE from linear to quadratic fit, but not for subsequent higher-degree polynomials.

The `outer()` method of the `np.power()` function takes in array arguments. It takes in two array arguments and forms a larger array by applying the `power` function to the first array argument sequantially with the values in the second array. This also works with the `add()` and `min()` functions from `np`.

In [26]:
A = np.array([3, 5, 9])
B = np.array([2, 5])
np.add.outer(A,B)

array([[ 5,  8],
       [ 7, 10],
       [11, 14]])

Thus the call `np.power.outer(H, np.arange(d+1))` applies incrementally increasing powers to the rows of `H`, resulting in a higher-order polynomial fit as `d` ranges from 1 to 5.

Next we use the `KFold()` function to partition the data into $K=10$ random groups. We use `random_state` to set the random seed as before.

In [30]:
cv_error = np.zeros(5)
cv = KFold(n_splits=10, shuffle=True, random_state=0)
for i, d in enumerate(range(1,6)):
    X = np.power.outer(H, np.arange(d+1))
    M_CV = cross_validate(M, X, Y, cv=cv)
    cv_error[i] = f"{np.mean(M_CV['test_score']):.4f}"
cv_error

array([24.2077, 19.1853, 19.2763, 19.4785, 19.1372])

While the values for the test MSE is slightly different for a 10-fold CV, we still see the significant drop as we did for LOOCV going from linear to quadratic. We also saw improvement gains in compute time (while technically there is a closed formula for LLOCV, the `cross_validate` function does not leverage that closed form).

Note that `cross_validate()` can take different splitting mechanisms as arguments. For example, we can use `ShuffleSplit()` to implement the validation set approach just as easily as $K$-fold cross-validation.

In [34]:
validation = ShuffleSplit(n_splits=1, test_size=196, random_state=0)
results = cross_validate(hp_model, Auto.drop(['mpg'], axis=1), Auto['mpg'], cv=validation)
print(f"{results['test_score'][0]:.4f}")

23.6166


We can estimate the variability in the test error by running the following:

In [35]:
validation = ShuffleSplit(n_splits=10, test_size=196, random_state=0)
results = cross_validate(hp_model, Auto.drop(['mpg'], axis=1), Auto['mpg'], cv=validation)
print(f"{results['test_score'].mean():.4f}, {results['test_score'].std():.4f}")

23.8022, 1.4218


As an aside, note that the standard deviation is not a valid estimate for the sampling variability of the mean test score or the individual scores as the randomly-selected training samples overlap and introduce correlations, but it does give an idea of the Monte Carlo variation incurred by picking different random folds. (For true understanding of the sampling variability of the mean test score we would need to take multiple random sample datasets and train using them, with them sampled independently. Since we're sampling from the same individual random sample when we do our validation splits, we're introducing a lot of covariance among the terms.)

### 5.3.3 The Bootstrap

This illustrates the use of the bootstrap in the example of Section 5.2, and an example involving estimating the accuracy of linear regression model on the `Auto` data set.

#### Estimating the Accuracy of a Statistic of Interest

The bootstrap approach can be applied in almost all situations. As a simple example we use the `Portfolio` data set to estimate the sampling variance of the parameter $\alpha$ from formula (5.7) given below:

$$ \hat{\alpha} = \frac{\hat{\sigma}^2_Y - \hat{\sigma}_{XY}}{\hat{\sigma}_X^2 + \hat{\sigma}_Y^2 - 2 \hat{\sigma}_{XY}}$$

We will use the function `alpha_func()` which takes in a dataframe `D` assumed to have columns `X` and `Y`, along with a vector `idx` indicating what observations to use to estimate $\alpha$.

In [37]:
Portfolio = load_data("Portfolio")
def alpha_func(D, idx):
    cov_ = np.cov(D[['X', 'Y']].iloc[idx], rowvar=False)
    return (
        (cov_[1,1] - cov_[0,1]) /
        (cov_[0,0] + cov_[1,1] - 2 * cov_[0,1])
    )

The following call estimates $\alpha$ using all 100 observations:

In [42]:
print(f"{alpha_func(Portfolio, range(Portfolio.shape[0])):.4f}")

0.5758


Next we randomly select 100 observations with replacement. This is equivalent to constructing a new bootstrap dataset and recomputing $\hat{\alpha}$ based on the new data set.

In [45]:
rng = np.random.default_rng(0)
print(f"{alpha_func(Portfolio, rng.choice(100, 100, replace=True)):.4f}")

0.6074


We generalize this to create a simple function `boot_SE()` for computing the bootstrap standard error for arbitrary functions that take only a data frame as an argument.

In [48]:
def boot_SE(func, D, n=None, B=1000, seed=0):
    rng = np.random.default_rng(seed)
    first_, second_ = 0, 0
    n = n or D.shape[0]
    for _ in range(B):
        idx = rng.choice(D.index, n, replace=True)
        value = func(D, idx)
        first_ += value
        second_ += value**2
    return np.sqrt(second_ / B - (first_ / B)**2)

Note that the formula within the `return` line is computing the variance of the $\hat{\alpha}$ using the formula $\text{Var}(X) = \mathbb{E}[X^2] - (\mathbb{E}[X])^2$, normalizing by the number of samples that we took. We use our function to estimate the accuracy of our estimate $\alpha$ using $B = 1000$ bootstrap replications.

In [50]:
alpha_SE = boot_SE(alpha_func, Portfolio, B=1000, seed=0)
print(f"{alpha_SE:.4f}")

0.0912


Thus we see that $\text{SE}(\hat{\alpha}) \approx 0.0912$.

#### Estimating the Accuracy of a Linear Regression Model

We can use the bootstrap to assess the variability of the coefficient estimates and predictions from a statistical learning method. Here we use it to assess the variability of $\beta_0$ and $\beta_1$, the intercept and slope terms for the linear regression model using `horsepower` to predict `mpg` in the `Auto` data set. We compare the estimates obtained using the bootstrap to those obtained using the formulas for $\text{SE}(\hat{\beta}_0)$ and $\text{SE}(\hat{\beta}_1)$ found in Section 3.1.2.

First we write a generic function `boot_OLS()` for bootstrapping a regression model that takes a formula to define the corresponding regression. We use the `clone()` function to make a copy of the formula that can be refit to the new dataframe. Thus any derived features such as those defined by `poly()` will be re-fit on the resampled data frame.

In [51]:
def boot_OLS(model_matrix, response, D, idx):
    D_ = D.loc[idx]
    Y_ = D_[response]
    X_ = clone(model_matrix).fit_transform(D_)
    return sm.OLS(Y_, X_).fit().params

Note this is not quite what is needed as the first argument to `boot_SE()`, as we want the first two arguments which specify the model to not change in the bootstrap process. We use the function `partial()` from the `functools` module to do this: take a function as argument, and freeze some of its arguments from left to right.

In [52]:
hp_func = partial(boot_OLS, MS(['horsepower']), 'mpg')

Checking the arguments of `hp_func()` shows it has two arguments, `D` and `idx`, and hence now it can be used as the first input to `boot_SE()`. We perform 10 bootstrap samples.

In [62]:
rng = np.random.default_rng(0)
np.array([hp_func(Auto, rng.choice(Auto.index, 392, replace=True)) for _ in range(10)])

array([[39.12226577, -0.1555926 ],
       [37.18648613, -0.13915813],
       [37.46989244, -0.14112749],
       [38.56723252, -0.14830116],
       [38.95495707, -0.15315141],
       [39.12563927, -0.15261044],
       [38.45763251, -0.14767251],
       [38.43372587, -0.15019447],
       [37.87581142, -0.1409544 ],
       [37.95949036, -0.1451333 ]])

Next we will use the `boot_SE()` function to compute the standard errors of 1,000 bootstrap estimates for the intercept and slope terms.

In [64]:
hp_se = boot_SE(hp_func, Auto, B=1000, seed=10)
hp_se

intercept     0.731176
horsepower    0.006092
dtype: float64

This indicates that the bootstrap estimate for $\text{SE}(\hat{\beta}_0)$ is 0.73, while the bootstrap estimate for $\text{SE}(\hat{\beta_1})$ is 0.0061. We can also use standard formulas to compute the standard errors for the regression coefficients in a linear model using the `summarize()` function from `ISLP.sm`.

In [65]:
hp_model.fit(Auto, Auto['mpg'])
model_se = summarize(hp_model.results_)['std err']
model_se

intercept     0.717
horsepower    0.006
Name: std err, dtype: float64

Note that they are relatively close together!

Finally, we get the bootstrap standard error estimates for fitting a quadratic model, giving us estimates for $\text{SE}(\hat{\beta}_0)$, $\text{SE}(\hat{\beta}_1)$, and $\text{SE}(\hat{\beta}_2)$.

In [72]:
quad_model = MS([poly('horsepower', 2, raw=True)])
quad_func = partial(boot_OLS, quad_model, 'mpg')
boot_SE(quad_func, Auto, B=1000)

intercept                                  1.538641
poly(horsepower, degree=2, raw=True)[0]    0.024696
poly(horsepower, degree=2, raw=True)[1]    0.000090
dtype: float64

We compare with the results for standard error using `sm.OLS()`.

In [73]:
M = sm.OLS(Auto['mpg'], quad_model.fit_transform(Auto))
summarize(M.fit())['std err']

intercept                                  1.800
poly(horsepower, degree=2, raw=True)[0]    0.031
poly(horsepower, degree=2, raw=True)[1]    0.000
Name: std err, dtype: float64